In [33]:
# !pip install --upgrade pip
# !pip install tabulate
# !pip install pandas
# !pip install sklearn
# !pip install IPython

In [34]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

from tabulate import tabulate

# Load the movies dataset with overviews
movies_df = pd.read_csv('themoviedb_with_overviews.csv')

# Fill missing overviews with empty strings to avoid errors during vectorization
movies_df['overview'] = movies_df['overview'].fillna('')


In [35]:
# Step 1: Create the TF-IDF matrix for movie overviews
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(movies_df['overview'])


In [36]:
# Step 2: Compute cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [37]:
# Function to get movie recommendations based on a given movie title, with ratings and similarity threshold
def recommend_movies(title, top_k_recommendations=5, similarity_threshold=0.2):
    # Check if the title exists in the dataset
    if title not in movies_df['title'].values:
        return None

    # Get the index of the movie that matches the title
    idx = movies_df[movies_df['title'] == title].index[0]

    # Get similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Filter movies based on the similarity threshold
    sim_scores = [(i, score) for i, score in sim_scores if score >= similarity_threshold and i != idx]

    # Sort by similarity score in descending order
    similar_movies = [(i, score) for i, score in sim_scores]
    similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)

    # Get the indices of the top-k most similar movies
    top_k_indices = [i[0] for i in similar_movies[:top_k_recommendations]]

    # Filter the movies DataFrame for the top-k indices and sort them by release date
    top_movies = movies_df.iloc[top_k_indices]
    top_movies = top_movies.sort_values(by='release_date', ascending=False)

    # Return movies with titles, overviews, ratings, genres, and release dates
    return top_movies[['title', 'overview', 'vote_average', 'genre_ids', 'release_date']]


In [38]:
# Example usage
title = "Deep Water"
recommendations = recommend_movies(title, top_k_recommendations=5, similarity_threshold=0)

if recommendations is None:
    print(f"Movie '{title}' not found in the dataset.")
else:
    # Display target movie and recommendations
    target_movie = movies_df[movies_df["title"] == title][["title", "overview", "vote_average", "genre_ids"]]
    print("Target Movie:")
    display(target_movie)  # This will display the target movie in the notebook
    print("Similar Movies:")
    display(recommendations)  # This will display the recommendations in the notebook


Target Movie:


,title,overview,vote_average,genre_ids
1291,Deep Water,A well-to-do husband who allows his wife to ha...,5.807,"[18, 9648, 53]"


Similar Movies:


,title,overview,vote_average,genre_ids,release_date
9076,Haseen Dillruba,Under investigation as a suspect in her husban...,6.597,"[9648, 53, 80, 10749]",2021-07-02
3297,The Swordsman,Young lovers brings two fighting clans together.,7.496,"[28, 18, 36]",2020-09-23
8992,The Chaser,"A wife, tired of her husband's non-stop carous...",7.759,"[80, 53, 28]",2008-02-14
2271,Basic Instinct,"Catherine, a novelist with an insatiable sexua...",6.918,"[53, 9648]",1992-03-20
2452,Emmanuelle: The Joys of a Woman,Emmanuelle returns to her husband in Hong Kong...,5.665,"[18, 10749]",1975-12-15
